# Q-Mini training pipeline
From-scratch, attention-free CausalConv-MoE model. No pretrained weights are downloaded. Keep datasets and checkpoints in Drive, not this public source repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA = Path('/content/drive/MyDrive/QMini/datasets')
ART = Path('/content/drive/MyDrive/QMini/artifacts')
ART.mkdir(parents=True, exist_ok=True)
print('Data:', DATA, '| Artifacts:', ART)


In [ ]:
import os, subprocess
REPO = Path('/content/Q-Mini')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/lion36508/Q-Mini.git',str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
subprocess.run(['python','-m','pip','install','-e','.[dev]'],check=True)
import torch
print('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))


In [ ]:
TOK = ART/'tokenizer'/'tokenizer.json'; CFG = ART/'qmini_active_config.json'
TOK.parent.mkdir(parents=True,exist_ok=True)
if not TOK.exists(): subprocess.run(['python','-m','qmini.cli','tokenizer','--data-dir',str(DATA),'--output',str(TOK),'--vocab-size','4096','--config-output',str(CFG)],check=True)
else: print('Existing tokenizer preserved:',TOK)
assert CFG.exists(), 'Active config is required and is never silently replaced.'


In [ ]:
TOKENS = ART/'tokens_v1'
if not (TOKENS/'train.bin').exists() and not (TOKENS/'valid.bin').exists():
    subprocess.run(['python','-m','qmini.cli','prepare','--data-dir',str(DATA),'--tokenizer',str(TOK),'--output-dir',str(TOKENS),'--valid-fraction','0.02','--loss-mask','auto'],check=True)
else: print('Existing token streams preserved:',TOKENS)
print('train tokens:',(TOKENS/'train.bin').stat().st_size//2,'valid tokens:',(TOKENS/'valid.bin').stat().st_size//2)


In [ ]:
SMOKE = ART/'runs'/'smoke'
if not (SMOKE/'latest.pt').exists():
    subprocess.run(['python','-m','qmini.cli','train','--config',str(REPO/'configs'/'qmini_smoke.json'),'--train-bin',str(TOKENS/'train.bin'),'--valid-bin',str(TOKENS/'valid.bin'),'--train-mask-bin',str(TOKENS/'train.mask.bin'),'--valid-mask-bin',str(TOKENS/'valid.mask.bin'),'--tokenizer',str(TOK),'--output-dir',str(SMOKE),'--steps','5','--batch-size','1','--seq-len','32','--grad-accum','1','--eval-every','5','--save-every','5','--eval-batches','2'],check=True)
else: print('Smoke checkpoint preserved:',SMOKE)


In [ ]:
BASE = ART/'runs'/'base'
args = ['python','-m','qmini.cli','train','--config',str(CFG),'--train-bin',str(TOKENS/'train.bin'),'--valid-bin',str(TOKENS/'valid.bin'),'--train-mask-bin',str(TOKENS/'train.mask.bin'),'--valid-mask-bin',str(TOKENS/'valid.mask.bin'),'--tokenizer',str(TOK),'--output-dir',str(BASE),'--steps','10000','--batch-size','2','--seq-len','512','--grad-accum','8','--eval-every','250','--save-every','250']
if (BASE/'latest.pt').exists(): args.append('--resume')
subprocess.run(args,check=True)
